# PicoRV32 + CIM SoC — PYNQ 板上串口验证 (pyserial 可用)

**运行位置**: PYNQ-Z2 Jupyter Notebook  
**前置条件**: pyserial 已安装 (`pip install pyserial`)

## 工作流程

1. 加载 `cim_rv32_soc.bit` (Pure-PL, 无需 hwh)
2. 通过 pyserial 读取 PicoRV32 UART 输出
3. 加载 `small_mlp_data/` 做 Golden Model 推理
4. 比较 UART 结果与 Golden Model

## 硬件连接

USB-TTL 适配器的 USB 端插入 **PYNQ 板的 USB 口** (不是 PC):

```
PYNQ-Z2 PMODA              USB-TTL            PYNQ USB
  Pin 1 (Y18, TX) -------> RXD                  |
  Pin 5 (GND)     -------> GND    ---USB--->  /dev/ttyUSB0
```

串口参数: **115200 baud, 8N1, 无流控**

## 上传文件清单

| 文件 | 说明 |
|------|------|
| `cim_rv32_soc.bit` | PicoRV32 版 bitstream (checkpoint2) |
| `small_mlp_data/` | 模型数据目录 (权重/偏置/测试图片/量化参数) |

## 1. 加载 Bitstream

In [ ]:
import subprocess, os, time, glob, struct
import numpy as np

BIT_FILE = "cim_rv32_soc.bit"

assert os.path.exists(BIT_FILE), f"{BIT_FILE} not found! Please upload."

print(f"Loading {BIT_FILE} ({os.path.getsize(BIT_FILE)/1024:.0f} KB)...")

try:
    subprocess.run(
        ["sudo", "bash", "-c", f"cat {BIT_FILE} > /dev/xdevcfg"],
        check=True, timeout=30
    )
    print("Bitstream loaded!")
except Exception as e:
    print(f"Method 1 failed: {e}")
    try:
        subprocess.run(["fpgautil", "-b", BIT_FILE], check=True, timeout=30)
        print("Bitstream loaded! (fpgautil)")
    except Exception as e2:
        print(f"Method 2 failed: {e2}")
        print("Manual: sudo bash -c 'cat cim_rv32_soc.bit > /dev/xdevcfg'")

print()
print("PicoRV32 is now executing firmware!")
print("  LED0 (R14): CIM done indicator")
print("  LED1 (P14): heartbeat (~1Hz blink = CPU running)")

## 2. 检测串口设备

In [ ]:
serial_ports = glob.glob("/dev/ttyUSB*") + glob.glob("/dev/ttyACM*")
print("Available serial ports on PYNQ:")
if serial_ports:
    for p in sorted(serial_ports):
        print(f"  {p}")
else:
    print("  No USB serial device detected!")
    print("  Check: USB-TTL adapter plugged into PYNQ USB port?")
    print("  Try: dmesg | tail")

## 3. 通过 pyserial 读取 UART 输出

运行此 Cell 后，按 **BTN0** 复位 PicoRV32 (如果 bitstream 刚加载则不需要)。

In [ ]:
import serial

# ========== CONFIG ==========
SERIAL_PORT = "/dev/ttyUSB0"   # Modify based on detection above
BAUD_RATE = 115200
TIMEOUT_SEC = 60
# =============================

print(f"Opening {SERIAL_PORT} @ {BAUD_RATE} baud...")
print("Tip: press BTN0 (reset) to restart PicoRV32 if no output\n")
print("-" * 60)

try:
    ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=1)
    ser.reset_input_buffer()

    uart_output = ""
    start_time = time.time()
    done = False

    while time.time() - start_time < TIMEOUT_SEC:
        raw = ser.readline()
        if raw:
            line = raw.decode(errors="ignore").rstrip("\r\n")
            if line:
                print(line)
                uart_output += line + "\n"
            if "=== Done ===" in line:
                done = True
                break

    ser.close()
    elapsed = time.time() - start_time
    print("-" * 60)

    if done:
        print(f"\nInference complete! Elapsed: {elapsed:.1f}s")
    else:
        print(f"\nTimeout ({TIMEOUT_SEC}s)! Received {len(uart_output)} chars")
        if not uart_output:
            print("No data received. Check:")
            print("  1. USB-TTL RXD connected to PMODA Pin 1 (Y18)")
            print("  2. GND connected")
            print(f"  3. Correct port? (current: {SERIAL_PORT})")
            print("  4. Press BTN0 to reset")

except serial.SerialException as e:
    print(f"Serial error: {e}")
    print(f"Confirm {SERIAL_PORT} exists and is not in use")
    uart_output = ""

## 4. 解析 UART 推理结果

In [ ]:
def parse_uart_output(text):
    """Parse PicoRV32 UART output into structured result."""
    result = {
        "predicted": None, "expected": None, "correct": None,
        "logits": [], "fc1_ok": False, "fc2_ok": False, "raw": text
    }
    for line in text.split("\n"):
        line = line.strip()
        if line.startswith("Predicted:"):
            try: result["predicted"] = int(line.split(":")[1].strip())
            except ValueError: pass
        elif line.startswith("Expected:"):
            try: result["expected"] = int(line.split(":")[1].strip())
            except ValueError: pass
        elif "CORRECT" in line:
            result["correct"] = True
        elif "WRONG" in line:
            result["correct"] = False
        elif line.startswith("Logits:"):
            nums = line.replace("Logits:", "").strip().split()
            result["logits"] = [int(x) for x in nums if x.lstrip("-").isdigit()]
        elif "FC1: done" in line:
            result["fc1_ok"] = True
        elif "FC2: computing" in line:
            result["fc2_ok"] = True
    return result

if uart_output:
    hw_result = parse_uart_output(uart_output)
    print("=" * 55)
    print("  PicoRV32 FPGA Inference Result (UART)")
    print("=" * 55)
    print(f"  FC1 (784->16, ReLU): {'OK' if hw_result['fc1_ok'] else 'N/A'}")
    print(f"  FC2 (16->10):        {'OK' if hw_result['fc2_ok'] else 'N/A'}")
    print(f"  Predicted: {hw_result['predicted']}")
    print(f"  Expected:  {hw_result['expected']}")
    status = 'CORRECT' if hw_result['correct'] else ('WRONG' if hw_result['correct'] is not None else 'UNKNOWN')
    print(f"  Result:    {status}")
    if hw_result["logits"]:
        print(f"  Logits:    {hw_result['logits']}")
        print(f"  ArgMax:    {np.argmax(hw_result['logits'])}")
    print("=" * 55)
else:
    hw_result = None
    print("No UART output to parse")

## 5. Golden Model 推理 (本地 small_mlp_data)

加载 `small_mlp_data/` 中的权重、偏置、量化参数和测试图片，
在 PYNQ 的 ARM PS 上做 bit-accurate INT8 推理，作为 golden reference。

In [ ]:
DATA_DIR = "small_mlp_data"

if not os.path.isdir(DATA_DIR):
    print(f"{DATA_DIR}/ not found! Please upload.")
    print("This directory is generated by prepare_picorv32_env.ipynb on host PC.")
    golden_results = None
else:
    # ---- Helper functions ----
    def read_hex_u32(path):
        with open(path) as f:
            return [int(l.strip(), 16) for l in f if l.strip()]

    def read_hex_u8(path):
        with open(path) as f:
            return [int(l.strip(), 16) & 0xFF for l in f if l.strip()]

    def unpack_weights(chunk_path, out_dim, in_dim):
        chunks = read_hex_u32(chunk_path)
        w = np.zeros((out_dim, in_dim), dtype=np.int8)
        idx = 0
        for ob in range((out_dim + 15) // 16):
            for ib in range((in_dim + 15) // 16):
                for ch in range(64):
                    r, cg = ch // 4, ch % 4
                    word = chunks[idx]; idx += 1
                    for b in range(4):
                        oi = ob * 16 + r
                        ii = ib * 16 + cg * 4 + b
                        if oi < out_dim and ii < in_dim:
                            val = (word >> (b * 8)) & 0xFF
                            w[oi, ii] = np.int8(struct.unpack('b', bytes([val]))[0])
        return w

    def unpack_bias(path):
        return np.array([np.int32(np.uint32(v)) for v in read_hex_u32(path)], dtype=np.int32)

    def hw_mvm(x_u8, w_i8, b_i32, zp, mult, shift, relu):
        """Bit-accurate INT8 MVM matching CIM hardware."""
        x_eff = np.clip(x_u8.astype(np.int32) - zp, 0, 511)
        acc = w_i8.astype(np.int32) @ x_eff + b_i32
        if relu:
            acc = np.maximum(acc, 0)
        out = np.zeros(len(acc), dtype=np.int32)
        for i in range(len(acc)):
            scaled = (acc[i] * mult) >> shift
            out[i] = max(-128, min(127, scaled))
        return out.astype(np.int8)

    # ---- Load model data ----
    qp = read_hex_u32(f"{DATA_DIR}/quant_params.hex")
    zps = read_hex_u32(f"{DATA_DIR}/zero_points.hex")
    fc1_mult, fc1_shift = qp[0], qp[1]
    fc2_mult, fc2_shift = qp[2], qp[3]
    hw_zp1 = np.int32(np.uint32(zps[0]))
    hw_zp2 = np.int32(np.uint32(zps[1]))

    w1 = unpack_weights(f"{DATA_DIR}/fc1_weight_tiles.hex", 16, 784)
    b1 = unpack_bias(f"{DATA_DIR}/fc1_bias.hex")
    w2 = unpack_weights(f"{DATA_DIR}/fc2_weight_tiles.hex", 10, 16)
    b2 = unpack_bias(f"{DATA_DIR}/fc2_bias.hex")

    print(f"Model loaded from {DATA_DIR}/")
    print(f"  FC1: [{w1.shape}], FC2: [{w2.shape}]")
    print(f"  Quant: fc1=({fc1_mult},{fc1_shift}), fc2=({fc2_mult},{fc2_shift})")
    print(f"  ZP: fc1={hw_zp1}, fc2={hw_zp2}")

    # ---- List test images ----
    img_dir = f"{DATA_DIR}/test_images"
    img_files = sorted(glob.glob(f"{img_dir}/img_*.hex"))
    n_images = len(img_files)
    print(f"  Test images: {n_images}")

    # ---- Run golden inference on all images ----
    golden_results = []
    for img_path in img_files:
        base = os.path.basename(img_path).replace(".hex", "")
        label_path = os.path.join(img_dir, f"{base}_label.txt")
        img_u8 = np.array(read_hex_u8(img_path), dtype=np.uint8)
        label = int(open(label_path).read().strip())

        fc1_out = hw_mvm(img_u8, w1, b1[:16], hw_zp1, fc1_mult, fc1_shift, relu=True)
        fc1_u8 = fc1_out.astype(np.uint8)
        fc2_out = hw_mvm(fc1_u8, w2, b2[:10], hw_zp2, fc2_mult, fc2_shift, relu=False)

        pred = int(np.argmax(fc2_out))
        logits = fc2_out.tolist()
        golden_results.append({
            "idx": base, "label": label, "pred": pred,
            "logits": logits, "correct": pred == label
        })

    g_correct = sum(1 for r in golden_results if r["correct"])
    print(f"\nGolden Model: {g_correct}/{n_images} correct ({100*g_correct/n_images:.0f}%)")
    print(f"{'Image':>12} {'Label':>6} {'Pred':>5} {'Result':>8}")
    print("-" * 40)
    for r in golden_results:
        status = "OK" if r["correct"] else "WRONG"
        print(f"{r['idx']:>12} {r['label']:6d} {r['pred']:5d} {status:>8}")

## 6. UART vs Golden Model 对比

将 UART 读取到的 PicoRV32 硬件推理结果与 Golden Model 进行比较。
因为固件只 bake 了一张图，所以按 `expected` label 匹配 golden 记录。

In [ ]:
if hw_result and hw_result["predicted"] is not None and golden_results:
    exp_label = hw_result["expected"]
    hw_pred = hw_result["predicted"]
    hw_logits = hw_result["logits"]

    # Find matching golden record by label
    matched = [r for r in golden_results if r["label"] == exp_label]

    print("=" * 60)
    print("  UART (PicoRV32 HW) vs Golden Model (Python INT8)")
    print("=" * 60)

    if matched:
        g = matched[0]
        print(f"  Image:     {g['idx']}  (label={exp_label})")
        print(f"  HW pred:   {hw_pred}")
        print(f"  Golden pred: {g['pred']}")
        print()

        # Compare prediction
        pred_match = hw_pred == g["pred"]

        # Compare logits if available
        logits_match = None
        if hw_logits and g["logits"]:
            logits_match = hw_logits == g["logits"]
            print(f"  HW logits:     {hw_logits}")
            print(f"  Golden logits: {g['logits']}")
            print()

        if pred_match and (logits_match is None or logits_match):
            print("  +----------------------------------------------------+")
            print("  |  BIT-EXACT MATCH: Hardware == Golden Model          |")
            print("  +----------------------------------------------------+")
        elif pred_match:
            print("  +----------------------------------------------------+")
            print("  |  PREDICTION MATCH (logits differ slightly)          |")
            print("  +----------------------------------------------------+")
        else:
            print("  +----------------------------------------------------+")
            print("  |  MISMATCH: HW pred != Golden pred                   |")
            print("  +----------------------------------------------------+")
    else:
        print(f"  No golden record found for label={exp_label}")

    print("=" * 60)

elif hw_result and hw_result["predicted"] is not None:
    print("No golden model data. Upload small_mlp_data/ to enable comparison.")
else:
    print("No UART inference result to compare.")

## 7. 重新运行 (reload + capture)

按 BTN0 或重新加载 bitstream，再次捕获 UART 输出。

In [ ]:
def reload_and_capture(bit_file=BIT_FILE, port=SERIAL_PORT,
                       baud=BAUD_RATE, timeout=TIMEOUT_SEC):
    """Reload bitstream, capture UART output, parse result."""
    print(f"Reloading {bit_file}...")
    subprocess.run(
        ["sudo", "bash", "-c", f"cat {bit_file} > /dev/xdevcfg"],
        check=True, timeout=30
    )
    time.sleep(0.5)

    ser = serial.Serial(port, baud, timeout=1)
    ser.reset_input_buffer()

    output = ""
    t0 = time.time()
    while time.time() - t0 < timeout:
        raw = ser.readline()
        if raw:
            line = raw.decode(errors="ignore").rstrip("\r\n")
            if line:
                print(line)
                output += line + "\n"
            if "=== Done ===" in line:
                break
    ser.close()
    return parse_uart_output(output)

# Uncomment to re-run:
# hw_result2 = reload_and_capture()

## 总结

本 notebook 在 **PYNQ 板上** 完成完整的验证闭环:

```
PYNQ PL (PicoRV32 + CIM) --UART--> USB-TTL --USB--> PYNQ PS (pyserial)
                                                         |
                                              Parse + Compare
                                                         |
PYNQ PS (Python golden model) <-- small_mlp_data/ -------+
```

不需要 PC 参与，整个验证在 PYNQ 上自包含完成。